# Question 7: Forecast August activity

This notebook starts from exported, model-ready data. It does not repeat cleaning. The objective is to forecast August visits, pounds distributed, and individuals served while enforcing a July 31 cutoff for every backtest and the final 2026 forecast.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd()
PIPELINE_DIR = CURRENT_DIR if CURRENT_DIR.name == "forecasting_pipeline" else CURRENT_DIR / "forecasting_pipeline"
DATA_DIR = PIPELINE_DIR / "data"
sys.path.insert(0, str(PIPELINE_DIR))

from cutoff_checks import assert_information_cutoff, assert_no_target_month

FINAL_CUTOFF = pd.Timestamp("2026-07-31")
FINAL_TARGET_MONTH = pd.Timestamp("2026-08-01")
PRIMARY_METRICS = ["visits", "pounds", "individuals"]

assert DATA_DIR.exists(), f"Export folder not found: {DATA_DIR}"

In [3]:
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load exported modeling data

In [4]:
def read_export(filename, date_columns=(), fips_columns=()):
    dtype = {column: "string" for column in fips_columns}
    frame = pd.read_csv(DATA_DIR / filename, dtype=dtype)
    for column in date_columns:
        if column in frame.columns:
            frame[column] = pd.to_datetime(frame[column], errors="coerce")
    return frame


food_site = read_export(
    "food_site_month_clean.csv", date_columns=["date", "month"], fips_columns=["fips"]
)
food_county = read_export(
    "food_county_month_clean.csv", date_columns=["date"], fips_columns=["fips"]
)
food_statewide = read_export(
    "food_statewide_month_clean.csv", date_columns=["date", "availability_date"]
)
mmg_county_year = read_export(
    "mmg_county_year_clean.csv", fips_columns=["fips"]
)
poverty_county_year = read_export(
    "poverty_county_year_clean.csv", fips_columns=["fips", "county_geoid"]
)
snap_agency_month = read_export(
    "snap_agency_month_clean.csv", date_columns=["date"]
)
snap_state_month = read_export(
    "snap_state_month_clean.csv", date_columns=["date", "availability_date"]
)
forecast_data = read_export(
    "forecast_ready_monthly.csv",
    date_columns=["date", "feature_cutoff", "date_snap", "availability_date_snap"],
)
cutoff_audit = read_export(
    "cutoff_audit.csv",
    date_columns=["cutoff", "latest_food_month", "latest_snap_month"],
)
imputation_validation = read_export("imputation_validation.csv")
imputation_validation_by_method = read_export("imputation_validation_by_method.csv")
food_county_outlier_review = read_export(
    "food_county_outlier_review.csv", date_columns=["date"], fips_columns=["fips"]
)
data_manifest = read_export(
    "data_manifest.csv", date_columns=["date_min", "date_max"]
)

data_manifest

,file,rows,columns,date_column,date_min,date_max,description
0,food_site_month_clean.csv,25399,57,date,2022-01-01,2026-07-01,Site-month food-shelf data with imputations an...
1,food_county_month_clean.csv,4785,52,date,2022-01-01,2026-07-01,"County-month food-shelf totals with reported, ..."
2,food_statewide_month_clean.csv,55,13,date,2022-01-01,2026-07-01,Statewide monthly reported and adjusted outcom...
3,mmg_county_year_clean.csv,261,51,NaN,NaT,NaT,MMG county-year data with robust outlier diagn...
4,poverty_county_year_clean.csv,261,43,NaN,NaT,NaT,ACS poverty county-year data with standardized...
5,snap_agency_month_clean.csv,4437,14,date,2022-01-01,2026-03-01,SNAP agency-month data with outlier diagnostic...
6,snap_state_month_clean.csv,51,7,date,2022-01-01,2026-03-01,Statewide SNAP totals with conservative availa...
7,forecast_ready_monthly.csv,55,34,date,2022-01-01,2026-07-01,Past-only statewide food-shelf lags and point-...
8,cutoff_audit.csv,4,6,cutoff,2023-07-31,2026-07-31,Available source periods at each July forecast...
9,imputation_validation.csv,3,6,NaN,NaT,NaT,Overall masked-value imputation validation by ...


## 2. Enforce readiness and cutoff checks

These checks must pass before any model is fitted.

In [5]:
assert food_county.duplicated(["fips", "date"]).sum() == 0
assert food_site.duplicated(["site_id", "date"]).sum() == 0
assert food_county["fips"].str.fullmatch(r"\d{5}").all()
assert food_county["fips"].nunique() == 87
assert food_county.groupby("fips")["date"].nunique().eq(55).all()
assert food_statewide["counties_in_grid"].eq(87).all()
assert food_statewide["counties_with_sites"].eq(86).all()

assert_information_cutoff(
    food_statewide,
    FINAL_CUTOFF,
    observation_date_col="date",
    availability_date_col="availability_date",
    frame_name="food_statewide",
)
assert_no_target_month(
    food_statewide,
    FINAL_TARGET_MONTH,
    date_col="date",
    frame_name="food_statewide",
)

snap_with_availability = snap_state_month.dropna(subset=["availability_date"])
assert_information_cutoff(
    snap_with_availability.loc[snap_with_availability["availability_date"].le(FINAL_CUTOFF)],
    FINAL_CUTOFF,
    observation_date_col="date",
    availability_date_col="availability_date",
    frame_name="snap_state_month",
)

readiness_summary = pd.DataFrame({
    "check": [
        "Maximum food-shelf month",
        "County-month duplicate keys",
        "Counties in grid",
        "Counties with supplied sites",
        "August 2026 outcome present",
    ],
    "result": [
        food_statewide["date"].max().date(),
        food_county.duplicated(["fips", "date"]).sum(),
        food_county["fips"].nunique(),
        int(food_statewide["counties_with_sites"].max()),
        food_statewide["date"].dt.to_period("M").eq(FINAL_TARGET_MONTH.to_period("M")).any(),
    ],
})
readiness_summary

,check,result
0,Maximum food-shelf month,2026-07-01
1,County-month duplicate keys,0
2,Counties in grid,87
3,Counties with supplied sites,86
4,August 2026 outcome present,False


## 3. Select the forecasting target

The reported series is the primary benchmark. The imputed series is a sensitivity analysis because masked-value imputation error was approximately 34–36% sMAPE.

In [6]:
TARGET_VERSION = "reported"  # Change to "imputed" only for sensitivity testing.
if TARGET_VERSION not in {"reported", "imputed"}:
    raise ValueError("TARGET_VERSION must be 'reported' or 'imputed'.")

target_columns = {
    metric: f"{metric}_{TARGET_VERSION}" for metric in PRIMARY_METRICS
}
model_targets = food_statewide[["date", *target_columns.values()]].rename(
    columns={column: metric for metric, column in target_columns.items()}
)
model_targets.tail()

,date,visits,pounds,individuals
50,2026-03-01,277840.0,1.359160e+07,862114.0
51,2026-04-01,263173.0,1.307424e+07,801510.0
52,2026-05-01,229370.0,1.153629e+07,710185.0
53,2026-06-01,239937.0,1.225381e+07,740478.0
54,2026-07-01,239143.0,1.299941e+07,739103.0


## 4. Rolling August backtest design

For each year, train using only information through July 31 and evaluate the August outcome. Primary comparison metric: MAPE. Also report MAE and WAPE because only three August backtests are available.

In [ ]:
BACKTEST_YEARS = [2023, 2024, 2025]
BACKTESTS = pd.DataFrame({
    "forecast_year": BACKTEST_YEARS,
    "cutoff": pd.to_datetime([f"{year}-07-31" for year in BACKTEST_YEARS]),
    "target_month": pd.to_datetime([f"{year}-08-01" for year in BACKTEST_YEARS]),
})

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = actual != 0
    return np.mean(np.abs((actual[valid] - predicted[valid]) / actual[valid])) * 100

def mae(actual, predicted):
    return np.mean(np.abs(np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)))

def wape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return np.abs(actual - predicted).sum() / np.abs(actual).sum() * 100

def rmse(actual, predicted):
    return np.sqrt(mean_squared_error(actual, predicted))

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(sparse=False, handle_unknown="ignore")

BACKTESTS

,forecast_year,cutoff,target_month
0,2023,2023-07-31,2023-08-01
1,2024,2024-07-31,2024-08-01
2,2025,2025-07-31,2025-08-01


## 5. Baseline models

Implement and compare:

1. Previous-August seasonal naive.
2. Year-to-date growth applied to previous August.
3. July multiplied by the historical August/July ratio.
4. Trend plus month-of-year regression.
5. Small robust ensemble.

## 6. Statistical and machine-learning challengers

Add challengers only after the baseline backtest is complete. Suitable candidates include regularized regression, robust regression, ETS or SARIMA, and a small gradient-boosted tree model using lagged features. Do not use LSTM or RNN as the primary approach with only 55 statewide monthly observations.

## 7. Final model selection and August 2026 forecast

Select a model based on held-out August performance, stability across all three outcomes, and compliance with the cutoff policy. Do not select a model only because it fits the historical series more closely.